In [1]:
import torch 
import os


/home/peppe/anaconda3/envs/my_env/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset_name = 'msmarco_data/msmarco_data_val_8' # 'bm25_1000_k_64_np32' / 'Oracolar_BM25_1000_K_64_TENS_DIM_F_64'

def retrieve_dataset_from_file(dataset_name):
    dataset = []
    path = f'../data/{dataset_name}/tensors/'
    list_of_files = os.listdir(path)
    print(len(list_of_files))
    for i, file in enumerate(list_of_files):
        print(file)
        diz = {}
        adj_matr = torch.load(path + file + '/adjacency_matrix.pt').type(torch.FloatTensor)
        doc_feat = torch.load(path + file + '/doc_feat_tensor_new.pt')
        qrels_tensor = torch.load(path + file + '/qrels_tensor_new.pt')
        query_tensor = torch.load(path + file + '/query_tensor_new.pt')
        qid = file
        print(qid)
        diz['doc_feat'] = doc_feat
        diz['adj_matrix'] = adj_matr
        diz['q_ids'] = qid
        diz['q_rels'] = qrels_tensor
        diz['query_feat'] = query_tensor

        if i > 100:
            break





        dataset.append(diz)
    return dataset


In [2]:
import pyterrier as pt
from pyterrier_dr import TctColBert, FlexIndex
if not pt.started():
    pt.init()
#dataset = pt.get_dataset("irds:cord19/trec-covid")


PyTerrier 0.10.0 has loaded Terrier 5.9 (built by craigm on 2024-05-02 17:40) and terrier-helper 0.0.8

No etc/terrier.properties, using terrier.default.properties for bootstrap configuration.


In [3]:
dataset = pt.get_dataset("irds:msmarco-passage")
model = TctColBert('castorini/tct_colbert-v2-hnp-msmarco')
index = FlexIndex('../data/msmarco-index_tctcolbert2/')
indexing_pipeline = model >> index
indexing_pipeline.index(dataset.get_corpus_iter())

# build a corpus graph from that index with 16 neighbours per document
index.corpus_graph(16)

msmarco-passage documents: 100%|██████████| 8841823/8841823 [5:28:59<00:00, 447.94it/s]  
indexing: 8841823dvec [5:28:59, 447.93dvec/s]
  0%|                                             | 0/583740 [00:00<?, ?chunk/s]/home/peppe/anaconda3/envs/my_env/lib/python3.10/site-packages/pyterrier_dr/flex/corpus_graph.py:50: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at  ../torch/csrc/utils/tensor_numpy.cpp:172.)
  left = torch.from_numpy(vectors[i*S:(i+1)*S]).to(device).to(dtype)
[INFO] [finished] [2:25:36] [583740chunk] [81.66chunk/s]                        
[2:25:36] [583740chunk] [66.82chunk/s]


NpTopKCorpusGraph('../data/msmarco-index_tctcolbert2/corpusgraph_k16', k=16)

: 

In [4]:
dataset_name = 'msmarco_data/test_graphs/'

data_normal = retrieve_dataset_from_file(dataset_name)


43
qid_1037798_tensors
qid_1037798_tensors
qid_1115776_tensors
qid_1115776_tensors
qid_182539_tensors
qid_182539_tensors
qid_47923_tensors
qid_47923_tensors
qid_1121709_tensors
qid_1121709_tensors
qid_443396_tensors
qid_443396_tensors
qid_1117099_tensors
qid_1117099_tensors
qid_527433_tensors
qid_527433_tensors
qid_131843_tensors
qid_131843_tensors
qid_1113437_tensors
qid_1113437_tensors
qid_207786_tensors
qid_207786_tensors
qid_915593_tensors
qid_915593_tensors
qid_1063750_tensors
qid_1063750_tensors
qid_168216_tensors
qid_168216_tensors
qid_1114646_tensors
qid_1114646_tensors
qid_148538_tensors
qid_148538_tensors
qid_833860_tensors
qid_833860_tensors
qid_855410_tensors
qid_855410_tensors
qid_1133167_tensors
qid_1133167_tensors
qid_1124210_tensors
qid_1124210_tensors
qid_405717_tensors
qid_405717_tensors
qid_1129237_tensors
qid_1129237_tensors
qid_264014_tensors
qid_264014_tensors
qid_1106007_tensors
qid_1106007_tensors
qid_146187_tensors
qid_146187_tensors
qid_87181_tensors
qid_87181

In [24]:
from torch_geometric.utils import subgraph

query = data_normal[30]['query_feat']
doc_feat = data_normal[30]['doc_feat']
adj_matrix = data_normal[30]['adj_matrix'].type(torch.LongTensor)

query_tot = torch.repeat_interleave(query, doc_feat.size()[0], dim=0)

scores = torch.sum(query_tot * doc_feat, dim = -1)

topk_stats = torch.topk(scores, k = 10, sorted = False)
values = topk_stats.values
indices = topk_stats.indices
# print(values)
# sg, _ = subgraph(indices, adj_matrix)
# print(adj_matrix.shape)



In [31]:
indices.dtype

torch.int64

In [6]:
import torch
from torch_geometric.data import Data
from torch_geometric.utils import degree

def count_isolated_nodes_coo(coo_tensor):
    # Convert COO tensor to a PyTorch Geometric Data object
    edge_index = coo_tensor
    data = Data(edge_index=edge_index)
    # Calculate the degree of each node
    degrees = degree(data.edge_index[0, :].type(torch.LongTensor), data.num_nodes)

    # Find isolated nodes (degree == 0)
    isolated_nodes = (degrees == 0).nonzero(as_tuple=False).squeeze()

    # Count the number of isolated nodes
    num_isolated_nodes = isolated_nodes.size(0)

    return num_isolated_nodes

count_isolated_nodes_coo(sg)

/home/peppe/anaconda3/envs/my_env/lib/python3.10/site-packages/torch_geometric/data/storage.py:304: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'edge_index'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  warnings.warn(


899

In [7]:
import torch
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from mpl_toolkits.mplot3d import Axes3D

def coo_to_adjacency_matrix(coo_tensor):
    row_indices = coo_tensor[0].long()
    col_indices = coo_tensor[1].long()
    size = (max(row_indices.max(), col_indices.max()) + 1, max(row_indices.max(), col_indices.max()) + 1)
    # Create a sparse COO tensor
    sparse_coo = torch.sparse_coo_tensor(indices=torch.stack([row_indices, col_indices]),
                                         values=torch.ones_like(row_indices).float(),
                                         size=size)
    # Convert the sparse COO tensor to a dense adjacency matrix
    adjacency_matrix = sparse_coo.to_dense()
    return adjacency_matrix


def plot_graph_from_adjacency(adjacency_tensor, directed=False):
    # Step 1: Convert the adjacency tensor to an adjacency matrix
    adjacency_matrix = coo_to_adjacency_matrix(adjacency_tensor)

    # Step 2: Identify isolated nodes
    isolated_nodes = torch.all(adjacency_matrix == 0, dim=0) & torch.all(adjacency_matrix == 0, dim=1)

    # Step 3: Remove isolated nodes from the adjacency matrix
    adjacency_matrix = adjacency_matrix#[~isolated_nodes][:, ~isolated_nodes]

    # Step 4: Create a graph using NetworkX
    if directed:
        G = nx.from_numpy_array(adjacency_matrix.numpy(), create_using=nx.DiGraph)
    else:
        G = nx.from_numpy_array(adjacency_matrix.numpy())

    # Step 5: Plot the graph with improved settings
    pos = nx.spring_layout(G, seed=42)  # Using spring_layout with a seed for reproducibility

    # Customize node colors, sizes, labels, etc., based on your requirements
    fig, ax = plt.subplots(figsize=(200, 100))  # Adjust figsize as needed
    nx.draw(G, pos, with_labels=True, node_color='skyblue', node_size=400, font_size=8, font_color='black', font_weight='bold', arrowsize=10 if directed else 0, ax=ax)

    plt.title("Graph Visualization")
    plt.show()

def plot_stuff(data_normal, aggr, n_to_show, plot = True):
    
    n_to_show = min(n_to_show, len(data_normal))

    
    edges_tensor = data_normal[n_to_show]['adj_matrix'] 
    docs = data_normal[n_to_show]['doc_feat']
    query_feat = data_normal[n_to_show]['query_feat']
    
    
    tsne = TSNE(n_components=3)
    x_tsne_3d = tsne.fit_transform(docs.detach().cpu().numpy())

    # Plot the t-SNE visualization in 3D
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(x_tsne_3d[:, 0], x_tsne_3d[:, 1], x_tsne_3d[:, 2])
    ax.set_title("t-SNE Visualization of Documents (3D)")
    ax.set_xlabel("Dimension 1")
    ax.set_ylabel("Dimension 2")
    ax.set_zlabel("Dimension 3")
    plt.show()

    if aggr == 'concat':
        rep_query = torch.repeat_interleave(query_feat, repeats=docs.shape[0], dim=0)
        print(rep_query.shape)
        docs = torch.cat((docs, rep_query), dim = -1)

    elif aggr == 'sum':

        rep_query = torch.repeat_interleave(query_feat, repeats=docs.shape[0], dim=0)
    
        # print(rep_query.shape)
        docs = docs + rep_query

    elif aggr == 'hadamard':
    
        rep_query = torch.repeat_interleave(query_feat, repeats=docs.shape[0], dim=0)
    
        # print(rep_query.shape)
        docs = docs * rep_query
    
    
    # Perform t-SNE dimensionality reduction
    tsne = TSNE(n_components=3)
    x_tsne_3d = tsne.fit_transform(docs.detach().cpu().numpy())

    # Plot the t-SNE visualization in 3D
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(x_tsne_3d[:, 0], x_tsne_3d[:, 1], x_tsne_3d[:, 2])
    ax.set_title("t-SNE Visualization of Documents (3D)")
    ax.set_xlabel("Dimension 1")
    ax.set_ylabel("Dimension 2")
    ax.set_zlabel("Dimension 3")
    plt.show()
    
    

    #Perform t-SNE dimensionality reduction
    if plot:
        plot_graph_from_adjacency(edges_tensor, directed=False)
    
# plot_stuff(data_normal, 'hadamard', 30, plot = True)
plot_graph_from_adjacency(sg, directed=False)


/home/peppe/anaconda3/envs/my_env/lib/python3.10/site-packages/networkx/drawing/nx_pylab.py:304: UserWarning: 

The arrowsize keyword argument is not applicable when drawing edges
with LineCollection.

To make this warning go away, either specify `arrows=True` to
force FancyArrowPatches or use the default value for arrowsize.
Note that using FancyArrowPatches may be slow for large graphs.

  draw_networkx_edges(G, pos, arrows=arrows, **edge_kwds)


In [ ]:
data_normal[0]['doc_feat'].shape


In [ ]:
def cosine_similarity_row_by_row(matrix1, matrix2):
    dot_product = torch.sum(matrix1 * matrix2, dim=1)
    norm_matrix1 = torch.norm(matrix1, dim=1)
    norm_matrix2 = torch.norm(matrix2, dim=1)

    similarity = dot_product / (norm_matrix1 * norm_matrix2 + 1e-8)  # Adding a small epsilon to avoid division by zero

    return similarity

out = cosine_similarity_row_by_row(x_reshaped, query_reshaped)

In [ ]:
from torchmetrics.functional.retrieval import retrieval_normalized_dcg


preds = torch.tensor([-0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.2663,
                     -0.2766,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607], device='cuda:0')

targets = torch.tensor([[2.],
                        [1.],
                        [1.],
                        [1.],
                        [2.],
                        [2.],
                        [1.],
                        [2.],
                        [2.],
                        [2.],
                        [1.],
                        [1.],
                        [2.],
                        [2.],
                        [2.],
                        [1.],
                        [2.],
                        [1.],
                        [1.],
                        [1.]])

# Flatten into one-dimensional vectors
preds = preds.view(-1)
targets = targets.view(-1)



ndcg = retrieval_normalized_dcg(preds, targets)
# You can use flat_preds and flat_targets as one-dimensional vectors in your code.


In [ ]:
import torch.nn.functional as F
import torch.nn as nn

class ListNetLoss(nn.Module):

    def __init__(self):
        super(ListNetLoss, self).__init__()

    def forward(self, y_pred, y_true, eps=1e-15):
        """
        ListNet loss introduced in "Learning to Rank: From Pairwise Approach to Listwise Approach".
        :param y_pred: predictions from the model, shape [batch_size, slate_length]
        :param y_true: ground truth labels, shape [batch_size, slate_length]
        :return: loss value, a torch.Tensor
        """
        y_pred = y_pred.clone()
        y_true = y_true.clone()

        preds_smax = F.softmax(y_pred, dim = 1)
        true_smax = F.softmax(y_true, dim = 1)

        preds_smax = preds_smax + eps
        preds_log = torch.log(preds_smax)

        return torch.mean(-torch.sum(true_smax * preds_log, dim=1))
    

class ListMLELoss(nn.Module):

    def __init__(self):
        super(ListMLELoss, self).__init__()

    def forward(self, y_pred, y_true, eps=1e-15, padded_value_indicator=-1.):
        
        """
        ListMLE loss introduced in "Listwise Approach to Learning to Rank - Theory and Algorithm".
        :param y_pred: predictions from the model, shape [batch_size, slate_length]
        :param y_true: ground truth labels, shape [batch_size, slate_length]
        :return: loss value, a torch.Tensor
        """

        y_pred = y_pred.clone().squeeze(-1).unsqueeze(0)
        y_true = y_true.clone().squeeze(-1).unsqueeze(0)

        print(y_pred.shape)
        print(y_true.shape)


        # shuffle for randomised tie resolution
        random_indices = torch.randperm(y_pred.shape[-1])
        y_pred_shuffled = y_pred[:, random_indices]
        y_true_shuffled = y_true[:, random_indices]

        y_true_sorted, indices = y_true_shuffled.sort(descending=True, dim=-1)

        mask = y_true_sorted == padded_value_indicator

        preds_sorted_by_true = torch.gather(y_pred_shuffled, dim=1, index=indices)
        preds_sorted_by_true[mask] = float("-inf")

        max_pred_values, _ = preds_sorted_by_true.max(dim=1, keepdim=True)

        preds_sorted_by_true_minus_max = preds_sorted_by_true - max_pred_values

        cumsums = torch.cumsum(preds_sorted_by_true_minus_max.exp().flip(dims=[1]), dim=1).flip(dims=[1])

        observation_loss = torch.log(cumsums + eps) - preds_sorted_by_true_minus_max

        observation_loss[mask] = 0.0

        return torch.mean(torch.sum(observation_loss, dim=1))




In [ ]:

preds = torch.tensor([10., 8., 4., 8., 8., 7., 4.])
target = torch.tensor([2., 2., 1., 2., 2., 2., 1.])

loss = ListMLELoss()

In [ ]:
loss(preds, target)

In [ ]:
import torch
from torch_geometric.nn import GINConv

# Define the input tensor
x = torch.tensor([[0.1, 0.2, 0.3],
                  [0.4, 0.5, 0.6],
                  [0.7, 0.8, 0.9]], dtype=torch.float)

# Define the adjacency matrix
edge_index = torch.tensor([[0, 1, 2],
                           [1, 2, 0]], dtype=torch.long)

# Define the GINConv layer
conv = GINConv(nn=torch.nn.Linear(3, 3))

# Apply the GINConv layer to the input tensor
output = conv(x, edge_index)

# Print the output tensor
print(output)


In [21]:
from pyterrier_dr import TctColBert, FlexIndex
import pyterrier as pt
if not pt.started(): 
    pt.init()
dataset = pt.get_dataset("irds:msmarco-passage")
model = TctColBert('castorini/tct_colbert-msmarco')
index = FlexIndex('./msmarco-index/')

# build a corpus graph from that index with 16 neighbours per document
index.corpus_graph(8)

NpTopKCorpusGraph('msmarco-index/corpusgraph_k16', k=8)

In [28]:
import torch 
ranges = torch.arange(0, 1000, 1)

rem = torch.randn(1, 768)

In [7]:
from torch_geometric.nn import SignedConv
from torch_geometric.utils import negative_sampling

import torch

conv = SignedConv(768, 768, first_aggr=True)



In [14]:
x = torch.randn(1000, 768)
pos_edge_index = torch.randint(0, 1000, (2, 8000))

neg_edge_index = negative_sampling(pos_edge_index, num_nodes=1000, num_neg_samples=8000)

In [10]:
out = conv(x, pos_edge_index, neg_edge_index)

In [15]:
neg_edge_index.shape

torch.Size([2, 8000])